# Notebook 04: Route Definition & Aggregation

## Objective
This notebook focuses on defining and aggregating logistics routes to measure shipping performance at the route level.

### Key Tasks:
- Define routes using Factory → State and Factory → Region
- Calculate total shipments (Route Volume)
- Calculate average shipping lead time
- Measure lead time variability
- Prepare route-level datasets for benchmarking

This step transforms order-level data into actionable route-level insights.

In [1]:
import pandas as pd
import numpy as np
import statistics as stats

pd.set_option("display.max_columns", None)

In [2]:
# Load feature-engineered dataset
file_path = r"../data/featured_nassau_data.csv"
ncd = pd.read_csv(file_path)

ncd.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Lead_Time_Days,Shipping Lead Time,Factory,Route_State,Route_Region,Order Month,Order Year,Order Month Name,Delay Status,Shipping Speed Category
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2024-01-09,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,6,6,Wicked Choccy's,Wicked Choccy's → Texas,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,7,7,Wicked Choccy's,Wicked Choccy's → Illinois,Wicked Choccy's → Interior,1,2024,January,Delayed,Slow
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2024-01-09,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,5,5,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,On-Time,Slow
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2024-01-11,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,7,7,Lot's O' Nuts,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior,1,2024,January,Delayed,Slow
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2024-01-12,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,7,7,Wicked Choccy's,Wicked Choccy's → Pennsylvania,Wicked Choccy's → Atlantic,1,2024,January,Delayed,Slow


In [4]:
# Ensure datetime format
ncd["Order Date"] = pd.to_datetime(ncd["Order Date"])
ncd["Ship Date"] = pd.to_datetime(ncd["Ship Date"])

## Route Aggregation (Factory → State)

Each route is defined as:
Factory → Customer State

We calculate:
- Total shipments (Route Volume)
- Average Lead Time
- Lead Time Variability

In [5]:
# Aggregate at Route_State level
route_state_summary = ncd.groupby("Route_State").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Std=("Shipping Lead Time", "std"),
    Min_Lead_Time=("Shipping Lead Time", "min"),
    Max_Lead_Time=("Shipping Lead Time", "max")
).reset_index()

In [6]:
# Round values for readability
route_state_summary["Avg_Lead_Time"] = route_state_summary["Avg_Lead_Time"].round(2)
route_state_summary["Lead_Time_Std"] = route_state_summary["Lead_Time_Std"].round(2)

route_state_summary.head()

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time
0,Lot's O' Nuts → Alabama,34,4.59,1.84,1,7
1,Lot's O' Nuts → Alberta,16,3.50,1.32,2,7
2,Lot's O' Nuts → Arizona,111,3.86,2.03,0,7
3,Lot's O' Nuts → Arkansas,31,4.19,1.54,0,7
4,Lot's O' Nuts → British Columbia,18,5.00,1.53,2,7


## Route Aggregation (Factory → Region)

Each route is defined as:
Factory → Customer Region

In [7]:
# Aggregate at Route_Region level
route_region_summary = ncd.groupby("Route_Region").agg(
    Total_Shipments=("Order ID", "count"),
    Avg_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Std=("Shipping Lead Time", "std"),
    Min_Lead_Time=("Shipping Lead Time", "min"),
    Max_Lead_Time=("Shipping Lead Time", "max")
).reset_index()

In [8]:
# Round values
route_region_summary["Avg_Lead_Time"] = route_region_summary["Avg_Lead_Time"].round(2)
route_region_summary["Lead_Time_Std"] = route_region_summary["Lead_Time_Std"].round(2)

route_region_summary.head()

,Route_Region,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time
0,Lot's O' Nuts → Atlantic,1661,3.93,1.87,0,7
1,Lot's O' Nuts → Gulf,897,4.10,1.83,0,7
2,Lot's O' Nuts → Interior,1321,4.09,1.82,0,7
3,Lot's O' Nuts → Pacific,1813,3.92,1.87,0,7
4,Secret Factory → Atlantic,72,4.04,1.81,1,7


In [9]:
# Top routes by volume
route_state_summary.sort_values(by="Total_Shipments", ascending=False).head(10)

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time
5,Lot's O' Nuts → California,1125,3.85,1.88,0,7
144,Wicked Choccy's → California,823,4.09,1.87,0,7
34,Lot's O' Nuts → New York,645,4.00,1.91,0,7
49,Lot's O' Nuts → Texas,569,4.14,1.86,0,7
173,Wicked Choccy's → New York,436,3.96,1.91,0,7
189,Wicked Choccy's → Texas,388,3.96,1.91,0,7
42,Lot's O' Nuts → Pennsylvania,331,4.17,1.78,0,7
53,Lot's O' Nuts → Washington,269,4.02,1.87,0,7
13,Lot's O' Nuts → Illinois,267,3.94,1.76,0,7
38,Lot's O' Nuts → Ohio,262,3.46,1.96,0,7


In [10]:
# Slowest routes
route_state_summary.sort_values(by="Avg_Lead_Time", ascending=False).head(10)

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time
133,The Other Factory → Rhode Island,1,7.00,NaN,7,7
124,The Other Factory → Missouri,1,7.00,NaN,7,7
127,The Other Factory → New Jersey,3,6.33,1.15,5,7
71,Secret Factory → Maryland,3,6.00,1.00,5,7
75,Secret Factory → Mississippi,1,6.00,NaN,6,6
99,Sugar Shack → Delaware,1,6.00,NaN,6,6
107,Sugar Shack → Rhode Island,1,6.00,NaN,6,6
78,Secret Factory → New Hampshire,1,6.00,NaN,6,6
138,The Other Factory → Washington,4,5.75,0.96,5,7
35,Lot's O' Nuts → Newfoundland And Labrador,6,5.67,1.63,3,7


In [11]:
# Fastest routes
route_state_summary.sort_values(by="Avg_Lead_Time").head(10)

,Route_State,Total_Shipments,Avg_Lead_Time,Lead_Time_Std,Min_Lead_Time,Max_Lead_Time
129,The Other Factory → North Carolina,1,0.00,NaN,0,0
120,The Other Factory → Kentucky,2,1.00,1.41,0,2
131,The Other Factory → Oregon,2,1.50,0.71,1,2
54,Lot's O' Nuts → West Virginia,2,1.50,2.12,0,3
112,The Other Factory → Arizona,2,2.00,0.00,2,2
125,The Other Factory → Nevada,1,2.00,NaN,2,2
126,The Other Factory → New Hampshire,2,2.00,0.00,2,2
108,Sugar Shack → Tennessee,1,2.00,NaN,2,2
80,Secret Factory → New Mexico,2,2.00,1.41,1,3
62,Secret Factory → Colorado,7,2.14,1.35,0,4


## Interpretation of Metrics

- Total Shipments → Route Volume
- Avg Lead Time → Efficiency indicator
- Lead Time Std → Consistency indicator
- Min/Max Lead Time → Range of performance

Routes with:
- High Avg Lead Time → inefficient
- High Std → inconsistent
- High Volume + High Lead Time → bottleneck

In [12]:
route_state_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Route_State      196 non-null    object 
 1   Total_Shipments  196 non-null    int64  
 2   Avg_Lead_Time    196 non-null    float64
 3   Lead_Time_Std    170 non-null    float64
 4   Min_Lead_Time    196 non-null    int64  
 5   Max_Lead_Time    196 non-null    int64  
dtypes: float64(2), int64(3), object(1)
memory usage: 9.3+ KB


In [13]:
route_region_summary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Route_Region     20 non-null     object 
 1   Total_Shipments  20 non-null     int64  
 2   Avg_Lead_Time    20 non-null     float64
 3   Lead_Time_Std    20 non-null     float64
 4   Min_Lead_Time    20 non-null     int64  
 5   Max_Lead_Time    20 non-null     int64  
dtypes: float64(2), int64(3), object(1)
memory usage: 1.1+ KB


In [14]:
# Save datasets
route_state_summary.to_csv("../data/route_state_summary.csv", index=False)
route_region_summary.to_csv("../data/route_region_summary.csv", index=False)

print("Route aggregation datasets saved successfully.")

Route aggregation datasets saved successfully.
